# Oracle4SD Atlas: from public oracle evidence to a checked economic episode

**Data version:** `uma-public-rpc-case-v1` · **Review bundle:** `atlas-review-20260916` · **Cutoff:** 2026-06-30 23:59:59 UTC.

This tutorial is for colleagues new to blockchain. It reconstructs **one real UMA request** from five transactions across Polygon and Ethereum. It does not download or certify the entire 105-million-row candidate resource. Use a CPU runtime with internet for setup and approximately 2 GB RAM for the case. The setup cell installs its own Python 3.12.12. No data-access key, wallet, GPU or payment is needed; full-data resources are documented separately. Package installation usually takes longer than the case calculation. Measured local results are supplied with the repository.

## 1. Overview and navigation

1. Overview and setup (this section)
2. Data sources and metadata
3. Querying
4. Queried-data inspection
5. Processing
6. Processed-data inspection
7. Analysis
8. Technical validation, full-data instructions and troubleshooting

An **oracle** is a mechanism for supplying claims to a blockchain program. A **transaction** is an action recorded by the chain; a **receipt** records its execution and events. **Collateral/principal** is a participant's own money committed to the mechanism. A reward is an incremental gain, so a payment returning principal is not entirely a reward. A protocol can resolve a dispute without proving external factual truth.

We follow the mentor's eight-part template, using its repository-preserved version recorded in `metadata/template_provenance.json`. Collection methodology is compared with Luo et al. (NeurIPS 2024), §3.2: [official paper](https://proceedings.neurips.cc/paper_files/paper/2024/file/3205b048f9cc54b9f7963db0b0f52d53-Paper-Datasets_and_Benchmarks_Track.pdf). We use protocol-native evidence and explicit selection; no GoG code or explorer fraud labels are used as truth.

### Setup

The next cell clones an immutable **code commit**, then installs a pinned managed Python 3.12 interpreter and creates a fresh isolated environment without using the host venv/ensurepip. A commit is a fixed repository revision. A virtual environment keeps dependency versions separate from the notebook service. All scientific commands use that environment's Python. The checkout path, exact revision and package freeze are retained. Run cells in order with **Runtime → Run all**; repeated execution creates a new run directory. Setup downloads the repository and pinned packages; later bundled replay is offline.

In [1]:
import csv, hashlib, json, os, platform, shutil, subprocess, sys, tempfile, time
from pathlib import Path
from IPython.display import display, Markdown
REPO_URL = os.environ.get("ORACLE_NOTEBOOK_REPO", "https://github.com/virusLuke3/Oracle4SD-Atlas.git")
CODE_REF = os.environ.get("ORACLE_NOTEBOOK_CODE_REF", "6317a1b66bc6f9653b21650a7d170dc762891a31")
DATA_VERSION = "uma-public-rpc-case-v1"
WORK = Path(tempfile.mkdtemp(prefix="oracle-atlas-tutorial-"))
REPO = WORK / "repository"
def command(args, cwd=None):
    child_env = os.environ.copy()
    child_env["MPLBACKEND"] = "Agg"  # Save figures without inheriting the host notebook backend.
    result = subprocess.run([str(a) for a in args], cwd=cwd, env=child_env, text=True, capture_output=True)
    with (WORK / "execution.log").open("a") as log:
        log.write(json.dumps([str(x) for x in args]) + "\n" + result.stdout + result.stderr + "\n")
    if result.returncode:
        print(result.stdout); print(result.stderr)
        raise RuntimeError("Command failed; see diagnostic output above")
    if result.stderr:
        print(result.stderr)
    return result.stdout
command(["git", "clone", "--quiet", "--no-checkout", REPO_URL, REPO])
command(["git", "checkout", "--quiet", "--detach", CODE_REF], cwd=REPO)
assert command(["git", "rev-parse", "HEAD"], cwd=REPO).strip() == CODE_REF
ENV = WORK / "managed-environment"
print(command([sys.executable, REPO / "scripts/bootstrap_environment.py", "--output", ENV])[-1600:])
PYTHON = ENV / "scientific" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
assert json.loads(command([PYTHON, "-c", "import json,sys; print(json.dumps(list(sys.version_info[:2])))"])) == [3, 12]
print(json.loads((ENV / "environment.json").read_text()))
RUN = REPO / "runs" / "tutorial"
RUN.mkdir(parents=True)
print({"code_commit": CODE_REF, "data_version": DATA_VERSION, "workspace": str(WORK)})

/tmp/oracle-atlas-tutorial-b_zsv7w8/managed-environment/scientific/bin/python

{'host_python': '3.12.7', 'host_executable': '/home/hy/develop/oracle-nature/release/Oracle4SD-Atlas/runs/revision-env-20261012/bin/python', 'scientific': {'python': '3.12.12', 'executable': '/tmp/oracle-atlas-tutorial-b_zsv7w8/managed-environment/scientific/bin/python'}, 'uv_version': '0.9.5', 'commands': [{'command': ['/home/hy/develop/oracle-nature/release/Oracle4SD-Atlas/runs/revision-env-20261012/bin/python', '-m', 'pip', 'install', '--disable-pip-version-check', '--no-deps', '--target', '/tmp/oracle-atlas-tutorial-b_zsv7w8/managed-environment/bootstrap-tools', 'uv==0.9.5'], 'exit_code': 0}, {'command': ['/tmp/oracle-atlas-tutorial-b_zsv7w8/managed-environment/bootstrap-tools/bin/uv', 'venv', '--python', '3.12.12', '--python-preference', 'only-managed', '--seed', '/tmp/oracle-atlas-tutorial-b_zsv7w8/managed-environment/scientific'], 'exit_code': 0}, {'command': ['/tmp/oracle-atlas-tutorial-b_zsv7w8/mana

## 2. Data sources and metadata

The fixed source specification names four Polygon transactions and one Ethereum transaction. A **block** is an ordered collection of transactions; a block hash identifies its exact contents. The source register lists external origins, licenses and file hashes. **SHA256** detects changed file bytes; it cannot prove a claim is factually true.

Read [metadata/source_register.csv](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/6317a1b66bc6f9653b21650a7d170dc762891a31/metadata/source_register.csv), [metadata/acquisition_profiles.json](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/6317a1b66bc6f9653b21650a7d170dc762891a31/metadata/acquisition_profiles.json) and [docs/API_VERIFICATION.md](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/6317a1b66bc6f9653b21650a7d170dc762891a31/docs/API_VERIFICATION.md). Official [Ethereum JSON-RPC](https://ethereum.org/developers/docs/apis/json-rpc/) and [Polygon endpoint documentation](https://docs.polygon.technology/pos/reference/rpc-endpoints) define the relevant calls. The larger project uses additional chain-native APIs, described in the acquisition profiles. Expected: seven source files, two chains, five transactions and an unchanged data version.

In [2]:
def read_json(path):
    return json.loads(Path(path).read_text())
cfg = read_json(REPO / "configs/project.json")
spec = read_json(REPO / "data/data_source/uma_case/case_spec.json")
assert cfg["data_version"] == DATA_VERSION
assert len(cfg["sources"]) == 7
for chain, details in spec["chains"].items():
    print(chain, "chain ID", details["chain_id"], "transactions", len(details["transactions"]))
for source in cfg["sources"]:
    data = (REPO / source["path"]).read_bytes()
    assert hashlib.sha256(data).hexdigest() == source["sha256"]
    print(source["source_id"], len(data), "bytes; SHA256 verified")
print("Expected source transactions:", sum(len(c["transactions"]) for c in spec["chains"].values()))

polygon chain ID 137 transactions 4
ethereum chain ID 1 transactions 1
case_spec.json 6089 bytes; SHA256 verified
raw/ethereum_rpc_snapshot.json 29216 bytes; SHA256 verified
raw/polygon_rpc_snapshot.json 99358 bytes; SHA256 verified
source/OptimisticOracleV2.sol 32293 bytes; SHA256 verified
source/OptimisticOracleV2Interface.sol 16138 bytes; SHA256 verified
source/VotingV2.abi.json 42629 bytes; SHA256 verified
source/VotingV2.sol 59632 bytes; SHA256 verified
Expected source transactions: 5


## 3. Querying

For a dependable first run we acquire the frozen public evidence already in the checkout. This **bundled mode** replays actual queried receipts; it makes no new API requests. The next cell invokes [code/query_data/query.py](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/6317a1b66bc6f9653b21650a7d170dc762891a31/code/query_data/query.py) and records every acquired file.

Optional separate runs may use `--mode public-download` (two immutable Hugging Face snapshots, checksum verified) or `--mode live` (five transaction hashes, chain ID, receipts, headers and code). Live mode has bounded retries and may fail when public archive access is unavailable. No pagination is needed for these fixed transaction hashes. Full history requires block ranges or cursor pagination; see the profiles. Expected: mode `bundled`, seven checked files, five source transactions.

In [3]:
def stage(folder, script, extra=()):
    return command([PYTHON, REPO / "code" / folder / script, "--config", REPO / "configs/project.json", "--run-dir", RUN, *extra], cwd=REPO)
stage("query_data", "query.py", ["--mode", "bundled"])
acquisition = read_json(RUN / "reports/acquisition.json")
assert acquisition["source_transactions"] == 5
assert all(f["frozen_hash_matches"] for f in acquisition["files"])
print({k: acquisition[k] for k in ["mode", "source_transactions", "network_scope", "failure_policy"]})

{'mode': 'bundled', 'source_transactions': 5, 'network_scope': 'no network', 'failure_policy': 'fail closed; no empty response imputed as zero'}


## 4. Queried-data inspection

Inspect the acquired receipts before deriving observations. A **log** is a contract event emitted in a transaction. A receipt may include logs unrelated to our selected economic flow; we do not count every log as a separate case. Expected: Polygon has four selected receipts and Ethereum one. The hash checks ensure the acquired copies match the registered inputs. Receipt success and exact canonical block matching are checked by the processing/QC stages.

In [4]:
for chain in ["polygon", "ethereum"]:
    snapshot = read_json(RUN / "queried_data/uma_case/raw" / f"{chain}_rpc_snapshot.json")
    print(chain, {"chain_id": snapshot["chain_id"], "receipts": len(snapshot["receipts"]), "receipt_log_count": sum(len(r["logs"]) for r in snapshot["receipts"].values())})
    assert len(snapshot["receipts"]) == len(spec["chains"][chain]["transactions"])
for entry in acquisition["files"]:
    assert hashlib.sha256((RUN / entry["path"]).read_bytes()).hexdigest() == entry["sha256"]
print("Acquired bytes match the recorded checksums.")

polygon {'chain_id': 137, 'receipts': 4, 'receipt_log_count': 29}
ethereum {'chain_id': 1, 'receipts': 1, 'receipt_log_count': 3}
Acquired bytes match the recorded checksums.


## 5. Processing

The processor decodes event bytes using the **ABI** (the contract interface describing event fields), links a request to its dispute and adjudication, and applies the pinned contract settlement rule. It retains integer token units to avoid rounding and distinguishes returned principal from rewards. One **episode** is the linked lifecycle, not a single transaction.

[code/process_data/process.py](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/6317a1b66bc6f9653b21650a7d170dc762891a31/code/process_data/process.py) calls the existing real-case builder; [docs/METHODS.md](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/6317a1b66bc6f9653b21650a7d170dc762891a31/docs/METHODS.md) explains each transformation. Expected: one episode, nine decoded Polygon events, eight USDC.e transfers and the retained QC/provenance tables. This stage does not infer missing off-chain costs or independent truth.

In [5]:
stage("process_data", "process.py")
processing = read_json(RUN / "reports/processing.json")
expected_counts = {"economic_episodes": 1, "source_events": 9, "token_flows": 8, "canonical_chain_qc": 30, "economic_link_qc": 13, "variable_provenance": 12, "lifecycle_timeline": 5, "economic_cashflows": 4, "abc_economic_measures": 28}
assert processing["counts"] == expected_counts
print(json.dumps(processing, indent=2))

{
  "amount_policy": "raw integers + asset + decimals; no cross-token sum",
  "counts": {
    "abc_economic_measures": 28,
    "canonical_chain_qc": 30,
    "economic_cashflows": 4,
    "economic_episodes": 1,
    "economic_link_qc": 13,
    "lifecycle_timeline": 5,
    "source_events": 9,
    "token_flows": 8,
    "variable_provenance": 12
  },
  "data_version": "uma-public-rpc-case-v1",
  "operation": "multi-event cross-chain linking -> one request lifecycle; transfers remain child rows",
  "rule_implementation": "code/production/scripts/case_studies/reproduce_uma_public_rpc_case.py",
  "scope": "one preregistered real request; no missing-value imputation",
  "source_transactions": 5
}


## 6. Processed-data inspection

The episode has 85 fields; the project's larger common table has a different versioned common schema. The [metadata/data_dictionary.csv](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/6317a1b66bc6f9653b21650a7d170dc762891a31/metadata/data_dictionary.csv) defines every case field, unit, null policy and example. A JSON **null** means unavailable or not identified, not zero. The 28 core A/B/C measures are also distinct from the dictionary's 33 variables, which include five support variables.

The next cell inspects the focal participant and source linkage, lists unavailable fields and confirms that the episode is settled but independent truth is absent. The child tables retain event-level provenance so a reader can trace the result back to receipts.

In [6]:
episode = read_json(RUN / "processed_data/economic_episode.json")
assert len(episode) == 85
print({key: episode[key] for key in ["episode_id", "actor_role", "terminal_outcome", "right_censored", "ground_truth_status", "source_transaction", "cross_chain_link_grade"]})
print("Unavailable fields:", [key for key, value in episode.items() if value is None])
print("Missingness explanation:", episode["missing_reason"])
with (RUN / "processed_data/lifecycle_timeline.csv").open() as handle:
    timeline = list(csv.DictReader(handle))
print("Lifecycle events:", len(timeline))
display(timeline)

{'episode_id': 'uma:polygon-oov2:0x699c0842ebca1553171853de9853d07f0336ca0909c90156e731b76442a875ca', 'actor_role': 'disputer', 'terminal_outcome': 'settled_disputed_disputer_wins', 'right_censored': False, 'ground_truth_status': 'protocol_resolution_only', 'source_transaction': '0xa5742b4d0b43dbe96de0e57480fbc11a6aa61c64765395db9f27ae45ae5b2198', 'cross_chain_link_grade': 'A'}
Unavailable fields: ['capital_cost_usd', 'delay_cost_usd', 'dvm_negative_slash_raw', 'dvm_positive_redistribution_raw', 'economic_regret_usd', 'independent_ground_truth', 'investigation_cost_usd', 'principal_slashed_raw', 'reward_forfeited_raw', 'usd_conversion_source', 'usd_conversion_time_unix', 'usd_value', 'verification_cost_usd']
Missingness explanation: Independent truth, off-chain investigation cost, timestamped USD conversion and DVM voter-level redistribution are outside this five-transaction public reproduction.
Lifecycle events: 5


[{'stage': 'Request',
  'timestamp': '1730232350',
  'chain': 'Polygon',
  'transaction_hash': '0x6fc39485f25f23d42bcb55baf68633b31aa5aa53a881116ecf1382e6a1245ede',
  'hours_since_request': '0.0'},
 {'stage': 'Proposal',
  'timestamp': '1730260036',
  'chain': 'Polygon',
  'transaction_hash': '0xa9ff6827a32b90f134345bbc646a480787676f481ac651b00088520379b47903',
  'hours_since_request': '7.690555555555555'},
 {'stage': 'Dispute',
  'timestamp': '1730260194',
  'chain': 'Polygon',
  'transaction_hash': '0xa5742b4d0b43dbe96de0e57480fbc11a6aa61c64765395db9f27ae45ae5b2198',
  'hours_since_request': '7.734444444444445'},
 {'stage': 'DVM resolution',
  'timestamp': '1730592035',
  'chain': 'Ethereum',
  'transaction_hash': '0xc845c4be3da71936e760b5fb8b70eeb8753a7bc5012c452e7d335166d9b14039',
  'hours_since_request': '99.9125'},
 {'stage': 'Settlement',
  'timestamp': '1730593355',
  'chain': 'Polygon',
  'transaction_hash': '0xe9457425a7a67a5e1bb431a7e3b1f8fe4649fcf04f3bb74376caa15600420310',

## 7. Analysis

This is a descriptive reconstruction of one settlement. USDC.e uses six decimals, so 1,000,000 raw units equal one token. The winner receives **1,000 USDC.e = 750 returned principal + 250 token gain**. The configured request reward of 5 USDC.e is a separate quantity. Native gas is a different asset and is not subtracted from USDC.e without a timestamped conversion.

The next cell runs [code/analyze_data/analyze.py](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/6317a1b66bc6f9653b21650a7d170dc762891a31/code/analyze_data/analyze.py), displays the accounting components and renders a simple table. It fits no model and supports no population estimate or claim of total net profit.

In [7]:
stage("analyze_data", "analyze.py")
summary = read_json(RUN / "analysis/summary.json")
scale = 10 ** episode["asset_decimals"]
principal = int(summary["principal_returned_raw"])
gain = int(summary["realized_payoff_raw"])
gross = int(summary["gross_payout_raw"])
assert (principal, gain, gross) == (750_000_000, 250_000_000, 1_000_000_000)
assert principal + gain == gross
display(Markdown("| Component | USDC.e |\n|---|---:|\n" + f"| Returned principal | {principal / scale:,.0f} |\n| Token gain | {gain / scale:,.0f} |\n| Gross transfer | {gross / scale:,.0f} |"))
print("Scope:", summary["scope"])

| Component | USDC.e |
|---|---:|
| Returned principal | 750 |
| Token gain | 250 |
| Gross transfer | 1,000 |

Scope: one case descriptive reconstruction; not population inference or full economic profit


## 8. Technical validation

Validation asks whether these particular bytes, identities, links and monetary rules are internally consistent. [code/technical_validation/validate.py](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/6317a1b66bc6f9653b21650a7d170dc762891a31/code/technical_validation/validate.py) checks source hashes, the 85-field schema, a preserved earlier summary, original raw-derived assertions and child QC. Expected: **28 top-level checks pass, zero fail**, with 30 canonical and 13 economic/link checks retained underneath. Do not add these levels and call the total independent tests.

The next cell executes validation and displays the denominator for each check. A failed check raises an error rather than silently producing a successful-looking report. The frozen summary is a regression reference, not independent external truth.

In [8]:
stage("technical_validation", "validate.py")
validation = read_json(RUN / "reports/validation.json")
assert validation["passed"] and validation["passed_checks"] == 28 and validation["failed_checks"] == 0
print({key: validation[key] for key in ["data_version", "passed_checks", "failed_checks", "not_evaluated"]})
display(validation["checks"])

{'data_version': 'uma-public-rpc-case-v1', 'passed_checks': 28, 'failed_checks': 0, 'not_evaluated': ['all-protocol recollection', 'population representativeness', 'independent external truth', 'full-core schema/application issues']}


[{'check': 'episode_schema',
  'denominator': 1,
  'passed': True,
  'scope': '85-field case schema; not full common schema',
  'unit': 'episode'},
 {'check': 'frozen_reference_values',
  'denominator': 1,
  'passed': True,
  'scope': 'previous reconstruction values; not independent factual truth',
  'unit': 'case'},
 {'check': 'episode_identity',
  'denominator': 1,
  'passed': True,
  'scope': 'fixed preregistered request',
  'unit': 'episode'},
 {'check': 'principal_not_profit',
  'denominator': 1,
  'passed': True,
  'scope': 'integer equality',
  'unit': 'settlement'},
 {'check': 'asset_precision',
  'denominator': 1,
  'passed': True,
  'scope': 'this case only',
  'unit': 'asset'},
 {'check': 'four_oov2_lifecycle_events',
  'denominator': 1,
  'passed': True,
  'scope': 'raw-derived case assertion',
  'unit': 'case assertion'},
 {'check': 'adapter_parameters_same_request',
  'denominator': 1,
  'passed': True,
  'scope': 'raw-derived case assertion',
  'unit': 'case assertion'},

### Deterministic rerun and provenance export

The next cell runs all four stages again in a new folder and compares the scientific summary. This checks deterministic reconstruction without reusing the first run's outputs; it is not independent scientific replication. It also runs negative tests for corrupted evidence, incorrect economic classifications and future-feature exclusion. The ZIP retains both runs, command logs, bootstrap logs, host/scientific package freezes and interpreter identities. Runtime metadata can differ; the scientific summary must match. A download link is displayed and hosted Colab offers a file download.


In [9]:
SECOND = REPO / "runs/determinism-repeat"
print(command([PYTHON, REPO / "scripts/check_repository.py"], cwd=REPO))
print(command([PYTHON, REPO / "scripts/run_pipeline.py", "--output", SECOND], cwd=REPO))
assert read_json(RUN / "reconstruction/statistics/summary.json") == read_json(SECOND / "reconstruction/statistics/summary.json")
print(command([PYTHON, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=REPO))
(RUN / "reports/environment.freeze.txt").write_text(command([PYTHON, "-m", "pip", "freeze"]))
(RUN / "reports/tutorial_identity.json").write_text(json.dumps({"code_commit": CODE_REF, "data_version": DATA_VERSION, "review_bundle": cfg["review_bundle"], "cutoff": cfg["fixed_cutoff"], "runtime_kind": "hosted_colab" if "google.colab" in sys.modules else "local_jupyter_kernel", "scope": "one real case; deterministic repeat; not full Atlas"}, indent=2))
EVIDENCE = WORK / "evidence"
EVIDENCE.mkdir()
shutil.copytree(RUN, EVIDENCE / "run1")
shutil.copytree(SECOND, EVIDENCE / "run2")
shutil.copy2(WORK / "execution.log", EVIDENCE / "execution.log")
for name in ["bootstrap.log", "environment.json", "scientific.freeze.txt"]:
    shutil.copy2(ENV / name, EVIDENCE / name)
(EVIDENCE / "host.freeze.txt").write_text(command([sys.executable, "-m", "pip", "freeze"]))
(EVIDENCE / "identity.json").write_text(json.dumps({"code_commit": CODE_REF, "data_version": DATA_VERSION, "host_python": platform.python_version(), "runtime_kind": "hosted_colab" if "google.colab" in sys.modules else "local_jupyter_kernel", "validation_kind": "same inputs and code: determinism replay, not independent scientific replication"}, indent=2))
export = shutil.make_archive(str(WORK / "oracle-atlas-case-evidence"), "zip", EVIDENCE)
print("Evidence ZIP:", export)
print("SUCCESS: two deterministic case replays; see separate full-release validation for the full dataset.")
from IPython.display import FileLink
display(FileLink(export))
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(export)

{"source_files": 7, "case_dictionary_fields": 85, "imported_code_files": 92, "local_links_valid": true}



query_data completed
process_data completed
analyze_data completed
technical_validation completed
{
  "run_dir": "/tmp/oracle-atlas-tutorial-b_zsv7w8/repository/runs/determinism-repeat",
  "data_version": "uma-public-rpc-case-v1",
  "validation_passed": true
}



/tmp/oracle-atlas-tutorial-b_zsv7w8/managed-environment/scientific/lib/python3.12/site-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/tmp/oracle-atlas-tutorial-b_zsv7w8/managed-environment/scientific/lib/python3.12/site-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/tmp/oracle-atlas-tutorial-b_zsv7w8/managed-environment/scientific/lib/python3.12/site-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/tmp/oracle-atlas-tutorial-b_zsv7w8/managed-environment/scientific/lib/python3.12/site-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /tmp/

Evidence ZIP: /tmp/oracle-atlas-tutorial-b_zsv7w8/oracle-atlas-case-evidence.zip
SUCCESS: two deterministic case replays; see separate full-release validation for the full dataset.


/tmp/oracle-atlas-tutorial-b_zsv7w8/oracle-atlas-case-evidence.zip

### Full dataset and resources

The frozen full-reconstruction component contains 64 native/curated input files (1,807,346,651 bytes), with immutable public download URLs and hashes. The versioned builders reconstruct the common, economic, research-sample and application layers. This notebook executes only the small case; inspect the separate full-release report for validation of the large layers.

See [full reconstruction commands](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/fix/usable-atlas-release-20261012/docs/FULL_DATA.md) and [revision evidence](https://github.com/virusLuke3/Oracle4SD-Atlas/tree/fix/usable-atlas-release-20261012/reports/revision_20261012). The historical Pyth snapshot is bundled and can be replayed offline. Five pinned upstream source/registry archives are downloadable. Allow at least 40 GB of disk for full reconstruction and use an adequately provisioned workstation/server. The original 15-table Hugging Face viewer remains a separate review subset. No DOI is assigned here.

### Troubleshooting

| Problem | Action |
|---|---|
| Managed interpreter/package download fails | Check internet access, retain bootstrap.log, then restart Run all |
| Host cannot import pip | Use a Python runtime with pip; the scientific interpreter is installed separately |
| Source checksum differs | Re-fetch the pinned revision; never change expected hashes to force a pass |
| Output directory already exists | Run setup again to allocate a new workspace |
| Live endpoint cannot serve archive state | Retain the failed run; use a suitable endpoint or select bundled replay |
| Kernel was interrupted | Restart and Run all; avoid partial stage outputs |
| A scientific assertion fails | Retain stage stderr and validation JSON for investigation |

Confirm the code commit, data version and cutoff; inspect payment decomposition and unknown values; check both validation reports and the exported logs. Local fresh-kernel execution is recorded separately from an actual hosted Google Colab session. Missing native timestamps, heterogeneous censoring and incomplete historical human provenance remain explicit limitations.
